# TechOps Intelligence Platform
## Notebook 06 — Retrieval Layer
**Phase:** 3 — Knowledge Base + Retrieval  
**Goal:** Build the hybrid retrieval layer that agents use
          to query the ChromaDB knowledge base

### What This Notebook Builds
1. BM25 keyword retrieval on all text corpora
2. Semantic retrieval from ChromaDB collections
3. Reciprocal Rank Fusion to merge both
4. Cross-encoder reranking for final top-k
5. Unified retrieval function used by all agents
6. Collection routing (which collection per query type)
7. Full evaluation of retrieval quality

### Why This Matters
Every agent decision is only as good as
what gets retrieved. This layer is the
most performance-critical component
in the entire system.

In [15]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version (compiled):", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))


Torch version: 2.5.1+cu121
CUDA available: True
CUDA version (compiled): 12.1
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [ ]:
import os
import re
import json
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from typing import List, Dict, Tuple, Optional
import numpy as np
import pandas as pd
from tqdm import tqdm

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi


In [ ]:
import chromadb
PROJECT_ROOT = Path("C:/Users/sudha/techops-intelligence")
os.chdir(PROJECT_ROOT)

PROCESSED  = PROJECT_ROOT / "data/processed"
EMBEDDINGS = PROJECT_ROOT / "data/embeddings"
from chromadb.config import Settings

# Reset any existing ChromaDB instances
chromadb.api.client.SharedSystemClient.clear_system_cache()

# Embedding model
print("Loading embedding model...")
embedding_model = SentenceTransformer(
    'sentence-transformers/all-mpnet-base-v2',
    device='cpu'
)
print("Embedding model ready")

# Cross-encoder for reranking
print("Loading cross-encoder reranker...")
cross_encoder = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-6-v2',
    device='cpu'
)
print("Cross-encoder ready")

# ChromaDB
chroma_path = str(EMBEDDINGS / "chroma_db")
client      = chromadb.PersistentClient(path=chroma_path, settings=chromadb.config.Settings(
    anonymized_telemetry=False))

# Load all collections
collections = {}
for name in ['incidents', 'postmortems', 'playbooks',
             'knowledge_base', 'logs', 'visuals']:
    try:
        collections[name] = client.get_collection(name)
        print(f"  {name:20} : {collections[name].count():,} docs")
    except Exception as e:
        print(f"  {name:20} : not found - {e}")

print(f"\nTotal collections loaded: {len(collections)}")

Loading embedding model...
Embedding model ready
Loading cross-encoder reranker...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Cross-encoder ready
  incidents            : 2,000 docs
  postmortems          : 175 docs
  playbooks            : 100 docs
  knowledge_base       : 11,975 docs
  logs                 : 1,120 docs
  visuals              : not found - Collection visuals does not exist.

Total collections loaded: 5


## 2. Build BM25 Index
BM25 is a keyword-based retrieval algorithm.
It excels at finding exact matches like error codes,
service names, port numbers, and hostnames.
We build one BM25 index per collection.

In [33]:
def tokenize_for_bm25(text: str) -> List[str]:
    """
    Tokenize text for BM25 indexing.
    Preserves IT-specific tokens like error codes,
    port numbers, and service names.
    """
    text   = text.lower()
    tokens = re.findall(r'[a-z0-9]+(?:[_\-\.][a-z0-9]+)*', text)
    return tokens


def build_bm25_index(collection, batch_size: int = 1000) -> Tuple:
    """
    Build BM25 index from all documents in a ChromaDB collection.
    Returns (bm25_index, document_list, id_list)
    """
    count = collection.count()
    if count == 0:
        return None, [], []

    all_docs = []
    all_ids  = []

    # Fetch in batches — ChromaDB has fetch limits
    offset = 0
    while offset < count:
        batch = collection.get(
            limit   = batch_size,
            offset  = offset,
            include = ['documents']
        )
        all_docs.extend(batch['documents'])
        all_ids.extend(batch['ids'])
        offset += batch_size

    # Tokenize for BM25
    tokenized = [tokenize_for_bm25(doc) for doc in all_docs]
    bm25      = BM25Okapi(tokenized)

    return bm25, all_docs, all_ids


# Build BM25 for each collection
# Skip visuals — image descriptions don't suit keyword search
bm25_indices = {}
print("Building BM25 indices...\n")

for name, collection in collections.items():
    if name == 'visuals':
        print(f"  {name:20} : skipped (vision content)")
        continue

    print(f"  Building BM25 for {name}...")
    bm25, docs, ids = build_bm25_index(collection)

    if bm25 is not None:
        bm25_indices[name] = {
            'bm25'  : bm25,
            'docs'  : docs,
            'ids'   : ids
        }
        print(f"  {name:20} : {len(docs):,} documents indexed")
    else:
        print(f"  {name:20} : empty - skipped")

print(f"\nBM25 indices built: {len(bm25_indices)}")

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Building BM25 indices...

  Building BM25 for incidents...
  incidents            : 2,000 documents indexed
  Building BM25 for postmortems...


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


  postmortems          : 175 documents indexed
  Building BM25 for playbooks...
  playbooks            : 100 documents indexed
  Building BM25 for knowledge_base...


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


  knowledge_base       : 11,975 documents indexed
  Building BM25 for logs...
  logs                 : 1,120 documents indexed

BM25 indices built: 5


## 3. Core Retrieval Functions
Three retrieval modes:
- BM25 keyword search
- Semantic vector search
- Hybrid search with Reciprocal Rank Fusion

In [34]:
def bm25_search(
    query          : str,
    collection_name: str,
    top_k          : int = 10
) -> List[Dict]:
    if collection_name not in bm25_indices:
        return []

    index  = bm25_indices[collection_name]
    tokens = tokenize_for_bm25(query)
    scores = index['bm25'].get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]

    results = []
    for rank, idx in enumerate(top_idx, 1):
        if scores[idx] > 0:
            # Fetch metadata from ChromaDB for this doc ID
            try:
                meta_fetch = collections[collection_name].get(
                    ids     = [index['ids'][idx]],
                    include = ['metadatas']
                )
                metadata = meta_fetch['metadatas'][0] \
                    if meta_fetch['metadatas'] else {}
            except Exception:
                metadata = {}

            results.append({
                'id'      : index['ids'][idx],
                'text'    : index['docs'][idx],
                'score'   : float(scores[idx]),
                'metadata': metadata,
                'source'  : 'bm25',
                'rank'    : rank
            })

    return results


def semantic_search(
    query          : str,
    collection_name: str,
    top_k          : int = 10,
    filter_metadata: Optional[Dict] = None
) -> List[Dict]:
    """
    Semantic vector retrieval from ChromaDB.
    Best for: conceptual queries, paraphrased questions.
    """
    if collection_name not in collections:
        return []

    query_embedding = embedding_model.encode([query]).tolist()

    query_kwargs = {
        'query_embeddings': query_embedding,
        'n_results'       : top_k,
        'include'         : ['documents', 'metadatas', 'distances']
    }

    if filter_metadata:
        query_kwargs['where'] = filter_metadata

    try:
        results_raw = collections[collection_name].query(
            **query_kwargs
        )
    except Exception as e:
        print(f"Semantic search error on {collection_name}: {e}")
        return []

    results = []
    for doc, meta, dist in zip(
        results_raw['documents'][0],
        results_raw['metadatas'][0],
        results_raw['distances'][0]
    ):
        results.append({
            'id'      : results_raw['ids'][0][len(results)],
            'text'    : doc,
            'score'   : float(1 - dist),
            'metadata': meta,
            'source'  : 'semantic',
            'rank'    : len(results) + 1
        })

    return results


def reciprocal_rank_fusion(
    bm25_results    : List[Dict],
    semantic_results: List[Dict],
    bm25_weight     : float = 0.4,
    semantic_weight : float = 0.6,
    k               : int   = 60
) -> List[Dict]:
    """
    Merge BM25 and semantic results using Reciprocal Rank Fusion.
    RRF score = weight / (k + rank)
    Higher score = more relevant.
    """
    fused_scores = {}
    all_docs     = {}

    # Score BM25 results
    for item in bm25_results:
        doc_id = item['id']
        rrf    = bm25_weight / (k + item['rank'])
        fused_scores[doc_id] = fused_scores.get(doc_id, 0) + rrf
        all_docs[doc_id]     = item

    # Score semantic results
    for item in semantic_results:
        doc_id = item['id']
        rrf    = semantic_weight / (k + item['rank'])
        fused_scores[doc_id] = fused_scores.get(doc_id, 0) + rrf
        if doc_id not in all_docs:
            all_docs[doc_id] = item

    # Sort by fused score
    sorted_ids = sorted(
        fused_scores.keys(),
        key=lambda x: fused_scores[x],
        reverse=True
    )

    fused_results = []
    for rank, doc_id in enumerate(sorted_ids, 1):
        item              = all_docs[doc_id].copy()
        item['rrf_score'] = fused_scores[doc_id]
        item['rank']      = rank
        fused_results.append(item)

    return fused_results


def rerank(
    query  : str,
    results: List[Dict],
    top_k  : int = 3
) -> List[Dict]:
    """
    Cross-encoder reranking of top candidates.
    More accurate than embedding similarity alone.
    Computes relevance score for each query-document pair.
    """
    if not results:
        return []

    # Prepare query-document pairs for cross-encoder
    pairs  = [(query, r['text'][:512]) for r in results]
    scores = cross_encoder.predict(pairs)

    # Attach cross-encoder scores
    for result, score in zip(results, scores):
        result['rerank_score'] = float(score)

    # Sort by rerank score
    reranked = sorted(
        results,
        key=lambda x: x['rerank_score'],
        reverse=True
    )[:top_k]

    for rank, item in enumerate(reranked, 1):
        item['final_rank'] = rank

    return reranked


print("Retrieval functions defined")

Retrieval functions defined



## 4. Collection Router
Determines which ChromaDB collections to search
based on the query type and agent context.
Different agents need different knowledge sources.

In [35]:
# Keywords that signal which collection to search
COLLECTION_ROUTING = {
    'incidents': [
        'incident', 'outage', 'down', 'failing', 'error',
        'timeout', 'connection refused', 'unreachable',
        'crashloop', 'spike', 'latency', 'degraded'
    ],
    'postmortems': [
        'root cause', 'why did', 'what caused', 'postmortem',
        'resolved', 'fix', 'similar incident', 'past incident',
        'investigation', 'diagnosis', 'happened before'
    ],
    'playbooks': [
        'playbook', 'runbook', 'how to respond', 'steps to',
        'procedure', 'protocol', 'what should i do',
        'ransomware', 'phishing', 'ddos', 'breach'
    ],
    'knowledge_base': [
        'slo', 'sla', 'error budget', 'reliability',
        'postmortem', 'on-call', 'escalation', 'runbook',
        'best practice', 'sre', 'how to handle',
        'troubleshoot', 'circuit breaker', 'capacity'
    ],
    'logs': [
        'log', 'metric', 'cpu', 'memory', 'disk', 'swap',
        'load average', 'iowait', 'anomaly', 'spike',
        'saturation', 'threshold', 'alert', 'monitoring'
    ]
}


def route_query(query: str) -> List[str]:
    """
    Determine which collections to search based on query content.
    Returns ordered list of collection names — most relevant first.
    """
    query_lower = query.lower()
    scores      = {}

    for collection, keywords in COLLECTION_ROUTING.items():
        if collection not in collections:
            continue
        score = sum(
            1 for kw in keywords
            if kw in query_lower
        )
        if score > 0:
            scores[collection] = score

    if not scores:
        # Default: search postmortems and knowledge_base
        return ['postmortems', 'knowledge_base']

    # Return sorted by relevance score
    return sorted(scores.keys(), key=lambda x: scores[x], reverse=True)


# Test routing
test_routing_queries = [
    "database connection refused port 5432",
    "how to respond to ransomware attack steps",
    "what is the SLO for payment service",
    "CPU spike anomaly on server system-4",
    "what caused the last memory exhaustion incident"
]

print("Collection routing test\n")
for q in test_routing_queries:
    routed = route_query(q)
    print(f"Query   : {q}")
    print(f"Routed  : {routed}")
    print()

Collection routing test

Query   : database connection refused port 5432
Routed  : ['incidents']

Query   : how to respond to ransomware attack steps
Routed  : ['playbooks']

Query   : what is the SLO for payment service
Routed  : ['knowledge_base']

Query   : CPU spike anomaly on server system-4
Routed  : ['logs', 'incidents']

Query   : what caused the last memory exhaustion incident
Routed  : ['incidents', 'postmortems', 'logs']



## 5. Unified Retrieval Function
Single entry point for all agent queries.
Handles routing, hybrid search, and reranking.
This is the function imported by all agents.

In [36]:
def retrieve(
    query           : str,
    collection_names: Optional[List[str]] = None,
    top_k_fetch     : int   = 10,
    top_k_return    : int   = 3,
    bm25_weight     : float = 0.4,
    semantic_weight : float = 0.6,
    use_reranking   : bool  = True,
    filter_metadata : Optional[Dict] = None
) -> List[Dict]:
    """
    Unified retrieval function for all TechOps agents.

    Pipeline:
    1. Route query to relevant collections
    2. BM25 keyword search per collection
    3. Semantic vector search per collection
    4. Reciprocal Rank Fusion merge
    5. Cross-encoder reranking
    6. Return top_k_return results

    Args:
        query           : natural language query from agent
        collection_names: override auto-routing if provided
        top_k_fetch     : candidates to fetch before reranking
        top_k_return    : final results to return after reranking
        bm25_weight     : weight for BM25 in RRF (0.0 to 1.0)
        semantic_weight : weight for semantic in RRF (0.0 to 1.0)
        use_reranking   : whether to apply cross-encoder reranking
        filter_metadata : optional ChromaDB metadata filter

    Returns:
        List of result dicts with text, score, metadata, source
    """
    # Step 1 - Route to collections
    if collection_names is None:
        collection_names = route_query(query)

    # Limit to max 2 collections per query for speed
    collection_names = collection_names[:2]

    all_bm25_results     = []
    all_semantic_results = []

    # Step 2+3 - Search each collection
    for col_name in collection_names:
        bm25_res = bm25_search(
            query, col_name, top_k=top_k_fetch
        )
        sem_res  = semantic_search(
            query, col_name,
            top_k=top_k_fetch,
            filter_metadata=filter_metadata
        )
        all_bm25_results.extend(bm25_res)
        all_semantic_results.extend(sem_res)

    if not all_bm25_results and not all_semantic_results:
        return []

    # Step 4 - Reciprocal Rank Fusion
    fused = reciprocal_rank_fusion(
        all_bm25_results,
        all_semantic_results,
        bm25_weight,
        semantic_weight
    )

    # Take top candidates for reranking
    candidates = fused[:min(top_k_fetch, len(fused))]

    # Step 5 - Reranking
    if use_reranking and candidates:
        final = rerank(query, candidates, top_k=top_k_return)
    else:
        final = candidates[:top_k_return]

    return final


def format_results(results: List[Dict]) -> str:
    """
    Format retrieval results for agent consumption.
    Returns structured string with ranked results.
    """
    if not results:
        return "No relevant documents found."

    output = []
    for i, result in enumerate(results, 1):
        score = result.get('rerank_score',
                result.get('rrf_score', 0))
        meta  = result.get('metadata', {})
        src   = meta.get('source', result.get('source', 'unknown'))

        output.append(
            f"Result {i} (score: {score:.3f}, source: {src}):\n"
            f"{result['text'][:400]}"
        )

    return "\n\n---\n\n".join(output)


print("Unified retrieval function defined")

Unified retrieval function defined


## 6. Retrieval Quality Evaluation
Test retrieval quality across all agent query types.
Measure precision at 3 for each category.

In [37]:
# Updated source mapping — what we actually store vs expected
SOURCE_TO_COLLECTION = {
    'synthetic_master' : 'incidents',
    'ftf_postmortem'   : 'postmortems',
    'ftf_playbook'     : 'playbooks',
    'ftf_runbook'      : 'knowledge_base',
    'ftf_sre_qa'       : 'knowledge_base',
    'ftf_metrics'      : 'logs',
}
# Add missing PDF sources to mapping
SOURCE_TO_COLLECTION.update({
    'site_reliability_engineering.pdf'           : 'knowledge_base',
    'building_secure_and_reliable_systems.pdf'   : 'knowledge_base',
    'aws_well_architected.pdf'                   : 'knowledge_base',
    'aws_genai_lens.pdf'                         : 'knowledge_base',
    'sre_workbook.pdf'                           : 'knowledge_base',
    'high_performance_sre.pdf'                   : 'knowledge_base',
    'pdf_knowledge'                              : 'knowledge_base',
})
def get_source(result: Dict) -> str:
    """Extract actual source from result — handles BM25 and semantic"""
    meta = result.get('metadata')
    if meta and isinstance(meta, dict):
        return meta.get('source', result.get('source', 'unknown'))
    return result.get('source', 'unknown')


def map_to_collection(source: str) -> str:
    """Map stored source value to collection name"""
    return SOURCE_TO_COLLECTION.get(source, source)


eval_queries = [
    {
        "query"       : "database connection refused PostgreSQL port 5432",
        "expected_col": ["incidents", "postmortems", "logs"],
        "category"    : "database_incident"
    },
    {
        "query"       : "kubernetes pod OOMKilled memory limit exceeded",
        "expected_col": ["incidents", "postmortems"],
        "category"    : "kubernetes_incident"
    },
    {
        "query"       : "how to handle on-call escalation during P1 incident",
        "expected_col": ["knowledge_base", "playbooks"],
        "category"    : "sre_process"
    },
    {
        "query"       : "ransomware attack response playbook steps",
        "expected_col": ["playbooks", "knowledge_base"],
        "category"    : "security_playbook"
    },
    {
        "query"       : "CPU saturation anomaly system load spike",
        "expected_col": ["logs", "postmortems"],
        "category"    : "metric_anomaly"
    },
    {
        "query"       : "error budget SLO reliability policy",
        "expected_col": ["knowledge_base"],
        "category"    : "sre_concept"
    },
    {
        "query"       : "disk full ENOSPC no space left on device",
        "expected_col": ["postmortems", "logs"],
        "category"    : "storage_incident"
    },
    {
        "query"       : "what caused the network DNS resolution failure",
        "expected_col": ["postmortems", "incidents"],
        "category"    : "diagnosis_query"
    },
    {
        "query"       : "blameless postmortem culture lessons learned",
        "expected_col": ["knowledge_base", "postmortems"],
        "category"    : "sre_culture"
    },
    {
        "query"       : "circuit breaker pattern cascading failure prevention",
        "expected_col": ["knowledge_base"],
        "category"    : "reliability_pattern"
    }
]

print("Full retrieval evaluation\n")
print(f"{'Category':<25} {'Source':<30} {'Collection':<15} {'Relevant'}")
print("-" * 80)

scores_all = []
relevant   = 0

for eval_item in eval_queries:
    results = retrieve(
        query         = eval_item['query'],
        top_k_fetch   = 10,
        top_k_return  = 3,
        use_reranking = True
    )

    if not results:
        print(f"  {eval_item['category']:<25} no results")
        continue

    top_result  = results[0]
    top_src     = get_source(top_result)
    top_col     = map_to_collection(top_src)
    top_score   = top_result.get(
        'rerank_score',
        top_result.get('rrf_score', 0)
    )

    is_relevant = top_col in eval_item['expected_col']
    if is_relevant:
        relevant += 1

    scores_all.append(top_score)

    print(
        f"  {eval_item['category']:<25} "
        f"{top_src:<30} "
        f"{top_col:<15} "
        f"{'yes' if is_relevant else 'no'}"
    )

print("-" * 80)
print(f"\nPrecision at 1  : {relevant}/{len(eval_queries)} "
      f"({relevant/len(eval_queries)*100:.0f}%)")

rerank_scores = [
    r.get('rerank_score', 0)
    for r in [retrieve(q['query'], top_k_return=1)[0]
              for q in eval_queries
              if retrieve(q['query'], top_k_return=1)]
    if r.get('rerank_score') is not None
]

if rerank_scores:
    print(f"Avg rerank score: {np.mean(rerank_scores):.3f}")
    print(f"Min rerank score: {min(rerank_scores):.3f}")
    print(f"Max rerank score: {max(rerank_scores):.3f}")

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Full retrieval evaluation

Category                  Source                         Collection      Relevant
--------------------------------------------------------------------------------


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  database_incident         synthetic_master               incidents       yes


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


...

  sre_concept               sre_workbook.pdf               knowledge_base  yes
  storage_incident          ftf_metrics                    logs            yes


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


  diagnosis_query           ftf_postmortem                 postmortems     yes
  sre_culture               site_reliability_engineering.pdf knowledge_base  yes
  reliability_pattern       site_reliability_engineering.pdf knowledge_base  yes
--------------------------------------------------------------------------------

Precision at 1  : 9/10 (90%)


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Avg rerank score: 1.687
Min rerank score: -11.452
Max rerank score: 10.057


In [38]:
# Save retrieval module to src/retrieval/
retrieval_module = '''"""
TechOps Intelligence Platform
Retrieval Layer — Hybrid Search + Reranking

Usage:
    from src.retrieval.retriever import retrieve, format_results

    results = retrieve(
        query        = "database connection refused",
        top_k_return = 3
    )
"""

import re
import json
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple, Optional

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import chromadb

PROJECT_ROOT = Path(__file__).resolve().parents[2]
EMBEDDINGS   = PROJECT_ROOT / "data/embeddings"

# ── Model Loading ──────────────────────────────────────
_embedding_model = None
_cross_encoder   = None
_client          = None
_collections     = {}
_bm25_indices    = {}


def _load_models():
    global _embedding_model, _cross_encoder, _client
    global _collections, _bm25_indices

    if _embedding_model is not None:
        return

    _embedding_model = SentenceTransformer(
        "sentence-transformers/all-mpnet-base-v2",
        device="cpu"
    )
    _cross_encoder = CrossEncoder(
        "cross-encoder/ms-marco-MiniLM-L-6-v2",
        device="cpu"
    )
    _client = chromadb.PersistentClient(
        path=str(EMBEDDINGS / "chroma_db")
    )

    for name in ["incidents", "postmortems", "playbooks",
                 "knowledge_base", "logs"]:
        try:
            _collections[name] = _client.get_collection(name)
        except Exception:
            pass

    for name, col in _collections.items():
        bm25, docs, ids = _build_bm25(col)
        if bm25:
            _bm25_indices[name] = {
                "bm25": bm25, "docs": docs, "ids": ids
            }


def _build_bm25(collection, batch_size=1000):
    count = collection.count()
    if count == 0:
        return None, [], []
    all_docs, all_ids = [], []
    offset = 0
    while offset < count:
        batch = collection.get(
            limit=batch_size, offset=offset,
            include=["documents"]
        )
        all_docs.extend(batch["documents"])
        all_ids.extend(batch["ids"])
        offset += batch_size
    tokenized = [_tokenize(d) for d in all_docs]
    return BM25Okapi(tokenized), all_docs, all_ids


def _tokenize(text: str) -> List[str]:
    return re.findall(r"[a-z0-9]+(?:[_\\-\\.][a-z0-9]+)*",
                      text.lower())


ROUTING = {
    "incidents"     : ["incident","outage","down","failing",
                       "error","timeout","crashloop","degraded"],
    "postmortems"   : ["root cause","what caused","why did",
                       "postmortem","fix","similar","diagnosis"],
    "playbooks"     : ["playbook","runbook","how to respond",
                       "procedure","ransomware","phishing","ddos"],
    "knowledge_base": ["slo","sla","error budget","reliability",
                       "best practice","sre","troubleshoot",
                       "circuit breaker","on-call","escalation"],
    "logs"          : ["log","metric","cpu","memory","disk",
                       "swap","load","iowait","anomaly","spike"]
}


def _route(query: str) -> List[str]:
    q      = query.lower()
    scores = {
        col: sum(1 for kw in kws if kw in q)
        for col, kws in ROUTING.items()
        if col in _collections
    }
    scored = [c for c, s in scores.items() if s > 0]
    if not scored:
        return ["postmortems", "knowledge_base"]
    return sorted(scored, key=lambda x: scores[x], reverse=True)


def retrieve(
    query           : str,
    collection_names: Optional[List[str]] = None,
    top_k_fetch     : int   = 10,
    top_k_return    : int   = 3,
    bm25_weight     : float = 0.4,
    semantic_weight : float = 0.6,
    use_reranking   : bool  = True,
    filter_metadata : Optional[Dict] = None
) -> List[Dict]:
    _load_models()

    cols = (collection_names or _route(query))[:2]
    bm25_res, sem_res = [], []

    for col in cols:
        if col in _bm25_indices:
            idx    = _bm25_indices[col]
            tokens = _tokenize(query)
            scores = idx["bm25"].get_scores(tokens)
            top_i  = np.argsort(scores)[::-1][:top_k_fetch]
            for rank, i in enumerate(top_i, 1):
                if scores[i] > 0:
                    bm25_res.append({
                        "id": idx["ids"][i], "text": idx["docs"][i],
                        "score": float(scores[i]),
                        "source": "bm25", "rank": rank
                    })

        if col in _collections:
            emb = _embedding_model.encode([query]).tolist()
            kw  = {"query_embeddings": emb, "n_results": top_k_fetch,
                   "include": ["documents", "metadatas", "distances"]}
            if filter_metadata:
                kw["where"] = filter_metadata
            try:
                r = _collections[col].query(**kw)
                for rank, (doc, meta, dist) in enumerate(zip(
                    r["documents"][0], r["metadatas"][0],
                    r["distances"][0]
                ), 1):
                    sem_res.append({
                        "id": r["ids"][0][rank-1], "text": doc,
                        "score": float(1 - dist), "metadata": meta,
                        "source": "semantic", "rank": rank
                    })
            except Exception:
                pass

    if not bm25_res and not sem_res:
        return []

    # RRF merge
    k=60
    fused, all_d = {}, {}
    for item in bm25_res:
        fused[item["id"]] = fused.get(item["id"],0) + bm25_weight/(k+item["rank"])
        all_d[item["id"]] = item
    for item in sem_res:
        fused[item["id"]] = fused.get(item["id"],0) + semantic_weight/(k+item["rank"])
        if item["id"] not in all_d:
            all_d[item["id"]] = item

    candidates = [
        {**all_d[id_], "rrf_score": s, "rank": r}
        for r, (id_, s) in enumerate(
            sorted(fused.items(), key=lambda x: -x[1]), 1
        )
    ][:top_k_fetch]

    if use_reranking and candidates:
        pairs  = [(query, c["text"][:512]) for c in candidates]
        rscores = _cross_encoder.predict(pairs)
        for c, s in zip(candidates, rscores):
            c["rerank_score"] = float(s)
        candidates = sorted(
            candidates, key=lambda x: x["rerank_score"], reverse=True
        )[:top_k_return]
        for i, c in enumerate(candidates, 1):
            c["final_rank"] = i

    return candidates[:top_k_return]


def format_results(results: List[Dict]) -> str:
    if not results:
        return "No relevant documents found."
    parts = []
    for i, r in enumerate(results, 1):
        score = r.get("rerank_score", r.get("rrf_score", 0))
        meta  = r.get("metadata", {})
        src   = meta.get("source", r.get("source", "unknown"))
        parts.append(
            f"Result {i} (score: {score:.3f}, source: {src}):\\n"
            f"{r['text'][:400]}"
        )
    return "\\n\\n---\\n\\n".join(parts)
'''

# Save to src/retrieval/retriever.py
retrieval_path = PROJECT_ROOT / "src/retrieval/retriever.py"
retrieval_path.parent.mkdir(parents=True, exist_ok=True)

with open(retrieval_path, 'w', encoding="utf-8") as f:
    f.write(retrieval_module)

# Create __init__.py
init_path = PROJECT_ROOT / "src/retrieval/__init__.py"
with open(init_path, 'w') as f:
    f.write("from .retriever import retrieve, format_results\n")

print(f"Retrieval module saved to: {retrieval_path}")
print("Agents can now import with:")
print("  from src.retrieval.retriever import retrieve, format_results")

Retrieval module saved to: C:\Users\sudha\techops-intelligence\src\retrieval\retriever.py
Agents can now import with:
  from src.retrieval.retriever import retrieve, format_results


In [39]:
print("NOTEBOOK 06 - RETRIEVAL LAYER COMPLETE")
print("=" * 55)

print("\nComponents built:")
print("  BM25 keyword search        - exact term matching")
print("  Semantic vector search     - meaning-based matching")
print("  Reciprocal Rank Fusion     - merges both results")
print("  Cross-encoder reranking    - precise relevance scoring")
print("  Collection router          - routes queries to right data")
print("  Unified retrieve() function - single agent entry point")

print(f"\nBM25 indices built for: {list(bm25_indices.keys())}")
print(f"Collections available  : {list(collections.keys())}")

print("\nRetrieval weights:")
print("  BM25     : 40%")
print("  Semantic : 60%")
print("  Reranker : ms-marco-MiniLM-L-6-v2")

print(f"\nPrecision at 1  : {relevant}/{len(eval_queries)} "
      f"({relevant/len(eval_queries)*100:.0f}%)")
print(f"Avg rerank score: {np.mean(scores_all):.3f}")

NOTEBOOK 06 - RETRIEVAL LAYER COMPLETE

Components built:
  BM25 keyword search        - exact term matching
  Semantic vector search     - meaning-based matching
  Reciprocal Rank Fusion     - merges both results
  Cross-encoder reranking    - precise relevance scoring
  Collection router          - routes queries to right data
  Unified retrieve() function - single agent entry point

BM25 indices built for: ['incidents', 'postmortems', 'playbooks', 'knowledge_base', 'logs']
Collections available  : ['incidents', 'postmortems', 'playbooks', 'knowledge_base', 'logs']

Retrieval weights:
  BM25     : 40%
  Semantic : 60%
  Reranker : ms-marco-MiniLM-L-6-v2

Precision at 1  : 9/10 (90%)
Avg rerank score: 1.687
